<a href="https://colab.research.google.com/github/mahima-lodhi/Relational-search-analytics-engine/blob/main/Ds1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =====================================================================
# PHASE 1: INITIALIZE SQL DATABASE & INGEST DATA
# =====================================================================
print("🗄️ Phase 1: Initializing Database...")
# Connect to a local SQLite file (it will create this file automatically)
conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()

# Drop table if it exists to allow fresh re-runs
cursor.execute("DROP TABLE IF EXISTS raw_inventory")

# Create a relational table
cursor.execute("""
CREATE TABLE raw_inventory (
    product_id INTEGER PRIMARY KEY,
    title TEXT,
    price_tag TEXT,
    description TEXT,
    category TEXT
)
""")

# Insert simulated messy enterprise inventory
messy_data = [
    (1, 'Ultra Lightweight Running Shoes', '$89.99', 'Perfect for rainy days and outdoor tracks.', 'Footwear'),
    (2, 'waterproof trail runner', '$120.00', 'Completely waterproof shell for hiking.', '  FOOTWEAR  '),
    (3, 'LightWeight gym shoes ', '75.50', 'Breathable mesh material.', 'footwear'),
    (4, 'Professional Marathon Sneaker', '   $150.00', 'Designed for high performance marathons.', 'Footwear'),
    (5, 'Casual Daily Loafers', None, 'Easy slip-on shoes for casual wear.', 'Footwear'),
    (6, 'Waterproof Winter Parka', '$210.00', 'Heavy insulated jacket for extreme cold rain.', 'Apparel'),
    (7, 'Breathable Running Tee', '$35.00', 'Moisture-wicking shirt for jogging.', '   APPAREL'),
    (8, 'Thermal Socks Pack', '15.99', 'Warm socks for winter sports.', 'apparel')
]

cursor.executemany('INSERT INTO raw_inventory VALUES (?, ?, ?, ?, ?)', messy_data)
conn.commit()

# =====================================================================
# PHASE 2: DATABASE-LEVEL CLEANING & ETL
# =====================================================================
print("🧹 Phase 2: Running SQL Cleaning Pipelines (CTEs)...")

sql_etl_query = """
WITH CalculatedMedian AS (
    -- Subquery to safely calculate average price for missing tags
    SELECT AVG(CAST(REPLACE(REPLACE(price_tag, '$', ''), ' ', '') AS REAL)) AS avg_price
    FROM raw_inventory
    WHERE price_tag IS NOT NULL
)
SELECT
    product_id,
    TRIM(LOWER(title)) AS title_clean,
    TRIM(LOWER(description)) AS description_clean,
    TRIM(LOWER(category)) AS category_clean,
    -- SQL Coalesce ensures no Null prices reach our analytics or models
    COALESCE(
        CAST(REPLACE(REPLACE(price_tag, '$', ''), ' ', '') AS REAL),
        (SELECT avg_price FROM CalculatedMedian)
    ) AS price_clean
FROM raw_inventory;
"""

# Extract the clean database view directly into Pandas
df_clean = pd.read_sql_query(sql_etl_query, conn)
conn.close() # Safely close database connection

# Combine text for the text matching engine
df_clean['search_metadata'] = df_clean['title_clean'] + " " + df_clean['description_clean']

# =====================================================================
# PHASE 3: EXPLORATORY VISUAL ANALYTICS
# =====================================================================
print("📊 Phase 3: Generating Business Analytics Charts...")
plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")

# Create a boxplot of prices grouped by clean categories
sns.boxplot(data=df_clean, x='category_clean', y='price_clean', palette='Set2', hue='category_clean', legend=False)
plt.title('Product Price Distribution by Cleaned Category', fontsize=14, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)

# Save chart automatically to the disk for your portfolio
plt.savefig('category_price_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("   💾 Success: Chart saved locally as 'category_price_distribution.png'")

# =====================================================================
# PHASE 4: AI VECTOR SEARCH ENGINE (TF-IDF)
# =====================================================================
print("🧠 Phase 4: Initializing Search Engine...")

def run_smart_search(query_text, data_frame):
    # Initialize Vectorizer
    tfidf = TfidfVectorizer()

    # Generate the math matrix from the inventory data
    tfidf_matrix = tfidf.fit_transform(data_frame['search_metadata'])

    # Transform user query to matching format
    query_vector = tfidf.transform([query_text.lower()])

    # Compute similarity scores against every single product
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Append scores and sort items
    results_df = data_frame.copy()
    results_df['match_score'] = scores
    return results_df.sort_values(by='match_score', ascending=False)

# Test the system with a real conversational query
sample_query = "running shoes for exercise"
search_output = run_smart_search(sample_query, df_clean)

print(f"\n🚀 SUCCESS: End-to-End Pipeline Complete.")
print(f"🔍 Top Search Results for: '{sample_query}'")
print("-" * 75)
for idx, row in search_output.head(3).iterrows():
    print(f"Match: {row['match_score']:.4f} | [{row['category_clean'].upper()}] {row['title_clean'].title()} | Price: ${row['price_clean']:.2f}")
print("-" * 75)

🗄️ Phase 1: Initializing Database...
🧹 Phase 2: Running SQL Cleaning Pipelines (CTEs)...
📊 Phase 3: Generating Business Analytics Charts...
   💾 Success: Chart saved locally as 'category_price_distribution.png'
🧠 Phase 4: Initializing Search Engine...

🚀 SUCCESS: End-to-End Pipeline Complete.
🔍 Top Search Results for: 'running shoes for exercise'
---------------------------------------------------------------------------
Match: 0.3951 | [FOOTWEAR] Ultra Lightweight Running Shoes | Price: $89.99
Match: 0.2940 | [APPAREL] Breathable Running Tee | Price: $35.00
Match: 0.1974 | [FOOTWEAR] Lightweight Gym Shoes | Price: $75.50
---------------------------------------------------------------------------


In [10]:
display(search_output)

,product_id,title_clean,description_clean,category_clean,price_clean,search_metadata,match_score
0,1,ultra lightweight running shoes,perfect for rainy days and outdoor tracks.,footwear,89.990000,ultra lightweight running shoes perfect for ra...,0.395083
6,7,breathable running tee,moisture-wicking shirt for jogging.,apparel,35.000000,breathable running tee moisture-wicking shirt ...,0.293950
2,3,lightweight gym shoes,breathable mesh material.,footwear,75.500000,lightweight gym shoes breathable mesh material.,0.197392
4,5,casual daily loafers,easy slip-on shoes for casual wear.,footwear,99.497143,casual daily loafers easy slip-on shoes for ca...,0.184800
3,4,professional marathon sneaker,designed for high performance marathons.,footwear,150.000000,professional marathon sneaker designed for hig...,0.062219
1,2,waterproof trail runner,completely waterproof shell for hiking.,footwear,120.000000,waterproof trail runner completely waterproof ...,0.058991
5,6,waterproof winter parka,heavy insulated jacket for extreme cold rain.,apparel,210.000000,waterproof winter parka heavy insulated jacket...,0.056914
7,8,thermal socks pack,warm socks for winter sports.,apparel,15.990000,thermal socks pack warm socks for winter sports.,0.055954


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
